# Training loss

Two models trained on **identical data** (same shards, same order, same batch size,
same LR schedule) — the only variable is size.

| | 124M | small |
|---|---|---|
| non-embedding params | 85.1M | 10.6M |
| throughput | 2,031 tok/s | 6,380 tok/s |
| tokens/param at 215M | 2.5 | 20.2 (Chinchilla) |

Every cell re-reads its log, so **re-running any single cell refreshes it** — safe
while a job is training.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys
# gpt2 is pip-installed (-e .); the plotting helper lives in scripts/
REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'scripts'))
import plot_loss as P

# committed copies in results/ so a fresh clone (e.g. Colab) works;
# point these at runs_small/log.txt or runs/log.txt to follow a live run
SMALL = os.path.join(REPO, 'results', 'log_small.txt')         # 30M model
BIG   = os.path.join(REPO, 'results', 'log_124M.txt')          # 124M model

LOG = SMALL          # which single-model curve the next two cells plot

# measured end-to-end seconds per step, for the wall-clock comparison
RATE = {'124M': 8.51, 'small': 2.62}
TOKENS_PER_STEP = 16384

for name, path in (('small', SMALL), ('124M', BIG)):
    d = P.read_log(path)
    print(f'{name:>6}: {len(d[0]):>6} train, {len(d[2]):>3} val, last step {int(d[0][-1]):>6}')

## One model's curve

Blue is train (raw per-step behind a rolling mean), orange is validation. The dashed
line is OpenAI's released GPT-2 124M on this same FineWeb-Edu val split.

Set `LOG = BIG` in the cell above to see the 124M instead.

In [ ]:
# Re-reads the log itself, so running THIS cell alone picks up a live run's latest
# steps. (Plotting the `data` variable from the setup cell would replot whatever was
# on disk when that cell last ran.)
fig = P.plot(*P.read_log(LOG), theme='light')   # theme='dark' for the dark version
fig

## The numbers

Every plotted value, readable without squinting at pixels.

In [ ]:
P.summarize(*P.read_log(LOG))   # re-read too, so this cell stands alone

## The comparison

Two views of the same two runs. They answer different questions and give opposite
answers — which is the whole point.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

c = P.THEMES['light']
MODELS = [('124M  (85.1M non-embd)', BIG,   RATE['124M'], c['train']),
          ('small (10.6M non-embd)', SMALL, RATE['small'], c['val'])]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=140)
fig.patch.set_facecolor(c['page'])

for ax, mode in zip(axes, ('tokens', 'hours')):
    ax.set_facecolor(c['surface'])
    ax.grid(axis='y', color=c['grid'], linewidth=0.8)
    ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(c['axis'])
    ax.tick_params(colors=c['muted'], labelsize=9, length=0)

    for label, path, rate, hue in MODELS:
        _, _, vs, vl = P.read_log(path)
        if not len(vs):
            continue
        x = vs * TOKENS_PER_STEP / 1e6 if mode == 'tokens' else vs * rate / 3600
        ax.plot(x, vl, color=hue, linewidth=2.0, marker='o', markersize=5,
                markeredgecolor=c['surface'], markeredgewidth=1.2, label=label)
        ax.annotate(f'{vl[-1]:.2f}', xy=(x[-1], vl[-1]), xytext=(7, 0),
                    textcoords='offset points', va='center', fontsize=9, color=c['ink2'])

    ax.axhline(P.GPT2_124M_BASELINE, color=c['muted'], linewidth=1.2, linestyle=(0, (5, 4)))
    ax.set_xlabel('Training tokens (M)' if mode == 'tokens' else 'Wall-clock hours',
                  fontsize=10, color=c['ink2'], labelpad=8)
    ax.set_title('Equal data' if mode == 'tokens' else 'Equal compute',
                 fontsize=12, color=c['ink'], loc='left', pad=26)

axes[0].set_ylabel('Validation loss', fontsize=10, color=c['ink2'], labelpad=8)
axes[0].text(0.995, P.GPT2_124M_BASELINE, f'  OpenAI GPT-2 (124M)  {P.GPT2_124M_BASELINE:.2f}  ',
             transform=axes[0].get_yaxis_transform(), ha='right', va='bottom',
             fontsize=8, color=c['muted'])

# one legend above the plots: inside the axes it collides with the endpoint labels
leg = axes[0].legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False,
                     fontsize=10, handlelength=1.6, borderaxespad=0)
for t in leg.get_texts():
    t.set_color(c['ink2'])
fig.tight_layout()

## The numbers behind it

The two views disagree because they hold different things constant.

In [ ]:
import math

def best(path):
    _, _, vs, vl = P.read_log(path)
    return (vl.min(), int(vs[vl.argmin()])) if len(vl) else (float('nan'), 0)

def val_near(path, step):
    _, _, vs, vl = P.read_log(path)
    return vl[np.abs(vs - step).argmin()]

sb, sstep = best(SMALL)
bb, bstep = best(BIG)
hours_small = sstep * RATE['small'] / 3600
big_at_same_hours = val_near(BIG, hours_small * 3600 / RATE['124M'])

print('EQUAL DATA')
print(f'  124M   val {bb:.4f}  ppl {math.exp(bb):>4.0f}   {bstep*RATE["124M"]/3600:>5.1f} h')
print(f'  small  val {sb:.4f}  ppl {math.exp(sb):>4.0f}   {hours_small:>5.1f} h')
print(f'  -> 124M better by {sb-bb:+.4f} nats, for {bstep*RATE["124M"]/3600/hours_small:.1f}x the compute')
print()
print(f'EQUAL COMPUTE ({hours_small:.1f} h)')
print(f'  124M   val {big_at_same_hours:.4f}  ppl {math.exp(big_at_same_hours):>4.0f}')
print(f'  small  val {sb:.4f}  ppl {math.exp(sb):>4.0f}')
print(f'  -> small better by {big_at_same_hours-sb:.4f} nats '
      f'({math.exp(big_at_same_hours)/math.exp(sb):.1f}x perplexity)')